In [12]:
"""
stage5_lgbm.py -- Stage 5: LGBM hazard on Hawkes-derived continuous features.

RULES
  S5.1  Target and timing identical to Stage 2 hazard: predict is_target[t]
        from info <= t-1. Row order identical to Featurizer.build for the same
        date range (same _selected / non-warm path), so metas align.
  S5.2  Per (stream, class): bars-since-last-event with event <= t-1
        (censored at session start -> local_t + 1), and exponentially decayed
        event counters s_t = sum_{k>=1} a^k * ind_{t-k}, a = exp(-1/tau),
        tau in TAUS (bars). Both use events <= t-1 only.
  S5.3  age (S2.3 definition) and tod passed as raw continuous columns.
  S5.4  Information set = pivot grammar only, matching Stage 2. Any gain over
        the calibrated Hawkes measures nonlinearity/interactions, not new data.
"""

import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression

from scipy.signal import lfilter
import lightgbm as lgb


TAUS = (2.0, 6.0, 18.0)
FRAME=6

TOD_START_MIN = 570      # first session minute (9:30)
TOD_BIN_MIN = 30
N_TOD = 13               # bins covering the session 6.5hrs / 30 min

STREAMS = ["MNQ_D1", "MNQ_D2", "TICK_D1", "TICK_D2"]

In [2]:
import os
print(os.getcwd())

/home/vm/pt/hgt-rl/mnq-tick


In [5]:
def _make_classes():
    cls = []
    for s in STREAMS:
        cls.append((s, "opp"))
        cls.append((s, "conf"))
    cls.append(("MNQ_JMA_SELF", "all"))
    return cls

def _bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins

def _age_bins_from_edges(edges):
    b = _bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b

class Featurizer:
    def __init__(self, bars, events,
                 bin_edges=(1, 2, 3, 5, 8, 13, 21, 34),
                 age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89)):
        self.bin_edges = list(bin_edges)
        self.age_edges = list(age_edges)
        self.bins = _bins_from_edges(bin_edges)
        self.age_bins = _age_bins_from_edges(age_edges)
        self.classes = _make_classes()
        self.feature_names = (
            [f"{s}|{c}|lag{lo}_{hi}" for (s, c) in self.classes for (lo, hi) in self.bins]
            + [f"age|{lo}_{hi}" for (lo, hi) in self.age_bins]
            + [f"tod|{k}" for k in range(N_TOD)]
        )
        self.n_feat = len(self.feature_names)

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]   # S2.6
        evg = dict(tuple(ev.groupby("date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"])
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - TOD_START_MIN   # S2.10
            tod = np.clip(mins // TOD_BIN_MIN, 0, N_TOD - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == "MNQ_JMA_SELF":
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))              # P[i] = count pos < i
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _fill(self, S, t, out):
        n = S["n"]
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:                                         # S2.1, S2.2
                b = np.clip(t - lo + 1, 0, n)
                a = np.clip(t - hi, 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S2.3
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][t]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _fill_frozen(self, S, q, h, out):
        """S2.5: features at t = q+h using only events/targets with pos <= q."""
        n = S["n"]
        t = q + h
        cap = q + 1
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:
                b = np.clip(np.minimum(t - lo + 1, cap), 0, n)
                a = np.clip(np.minimum(t - hi, cap), 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = S["lt_incl"][q]
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][np.minimum(t, n - 1)]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S

    def build(self, date_from=None, date_to=None):
        sel = []
        total = 0
        for S in self._selected(date_from, date_to):
            t = np.nonzero(~S["warm"])[0]                                      # S2.7
            sel.append((S, t))
            total += len(t)
        X = np.zeros((total, self.n_feat), np.float32)
        y = np.zeros(total, np.int8)
        bar_index = np.zeros(total, np.int64)
        sess_id = np.zeros(total, np.int32)
        timestamp = np.zeros(total, "datetime64[ns]")
        ofs = 0
        for sid, (S, t) in enumerate(sel):
            m = len(t)
            self._fill(S, t, X[ofs:ofs + m])
            y[ofs:ofs + m] = S["tgt"][t]
            bar_index[ofs:ofs + m] = S["bar_index"][t]
            sess_id[ofs:ofs + m] = sid
            timestamp[ofs:ofs + m] = S["timestamp"][t]
            ofs += m
        meta = pd.DataFrame({"bar_index": bar_index, "timestamp": timestamp,
                             "sess_id": sess_id, "is_target": y.astype(bool)})
        return X, y, meta

In [3]:
def build_lgbm_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)            # 0/1 event indicator per bar
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))     # last event <= t-1   S5.2
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))          # events shifted to <= t-1
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)             # s_t = a*(s_{t-1} + ind_{t-1})
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S5.3
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))

    names = []
    for (s, c) in fz.classes:
        names.append(f"{s}|{c}|bsince")
        names += [f"{s}|{c}|ewm{tau:g}" for tau in TAUS]
    names += ["age", "tod"]
    return np.concatenate(blocks), names


def build_target(fz, date_from=None, date_to=None):
    ys = [S["tgt"][np.nonzero(~S["warm"])[0]] for S in fz._selected(date_from, date_to)]
    return np.concatenate(ys).astype(np.int8)

In [10]:
bars = pd.read_parquet(f'data/stage-0/bars_{FRAME}s.parquet')
events = pd.read_parquet(f'data/stage-0/events_{FRAME}s.parquet')

fz = Featurizer(bars, events, bin_edges=(1, 2, 3, 5, 8, 13, 21, 34), age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89))

In [13]:
Xtr5, names = build_lgbm_features(fz, None, "2024-06-30")
Xva5, _ = build_lgbm_features(fz, "2024-07-01", "2024-12-31")
Xte5, _ = build_lgbm_features(fz, "2025-01-01", None)
ytr5 = build_target(fz, None, "2024-06-30")
yva5 = build_target(fz, "2024-07-01", "2024-12-31")
yte5 = build_target(fz, "2025-01-01", None)

params = dict(objective="binary", metric="binary_logloss", learning_rate=0.05,
              num_leaves=63, min_data_in_leaf=500, feature_fraction=0.9,
              bagging_fraction=0.8, bagging_freq=1, num_threads=16, verbose=-1)
dtr = lgb.Dataset(Xtr5, ytr5, feature_name=names)
dva = lgb.Dataset(Xva5, yva5, reference=dtr, feature_name=names)
m5 = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
               callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

p5 = m5.predict(Xte5, num_iteration=m5.best_iteration)
print("lgbm logloss", log_loss(yte5, p5))            # vs 0.3021
print("lgbm auc    ", roc_auc_score(yte5, p5))       # vs 0.714

pd.Series(m5.feature_importance("gain"), index=names).sort_values(ascending=False).head(15)

Training until validation scores don't improve for 100 rounds
[200]	valid_0's binary_logloss: 0.276931
[400]	valid_0's binary_logloss: 0.276466
[600]	valid_0's binary_logloss: 0.276302
[800]	valid_0's binary_logloss: 0.276269
Early stopping, best iteration is:
[804]	valid_0's binary_logloss: 0.276264
lgbm logloss 0.2740706096724571
lgbm auc     0.8006131563976401


MNQ_JMA_SELF|all|bsince    449778.972315
MNQ_D1|conf|ewm2           386645.887517
MNQ_D1|opp|bsince          357056.623092
MNQ_D1|opp|ewm2            317225.180622
MNQ_D1|conf|bsince         274835.840316
MNQ_JMA_SELF|all|ewm2      271050.478441
MNQ_D2|opp|ewm2             97971.170852
MNQ_D2|conf|ewm2            57947.521981
TICK_D1|opp|ewm2            55918.797176
MNQ_D2|opp|bsince           47212.138843
TICK_D1|conf|ewm2           39624.171354
TICK_D2|opp|ewm2            33639.069628
MNQ_D1|opp|ewm6             25916.800620
MNQ_JMA_SELF|all|ewm6       25765.580547
TICK_D2|conf|ewm2           25616.847334
dtype: float64